# 01 · Ingestion — WDBC Breast Cancer (Tier A)

**Stage 1 of the [STRUCTURE.md](../DOCS/STRUCTURE.md) pipeline.** Acquire the raw file into the analysis
environment *immutably*, log its lineage, validate it against the data dictionary, and classify it for
governance — **before** a single cleaning decision.

> **Business question (Stage 0):** *Can nuclear-morphology features from an FNA image reliably distinguish
> malignant from benign breast masses — and which measurements carry the diagnostic signal, given the 30
> features are heavily redundant by construction?*
> **Path:** A (diagnostic) → B (predictive classification). Path C (causal) is out of scope.
> **Positive class by cost:** Malignant (a false negative is the harmful error).

*This notebook imports no local modules — styling and contracts are inlined so a clone runs on `data.csv`
plus the pinned packages alone.*

In [1]:
# --- DESIGN.md palette + matplotlib style (inlined; notebooks import no local modules) ---
import matplotlib as mpl
import matplotlib.pyplot as plt

NAVY   = "#051C2C"   # ink only, never a series fill
BLUE   = "#2251FF"   # McKinsey blue -> emphasis / Malignant (at-risk class)
TEAL   = "#00857C"   # secondary series -> Benign
CYAN   = "#00A9F4"   # tertiary categorical
AMBER  = "#C1841C"   # reference / decision-threshold lines
SLATE  = "#7F93A6"   # muted labels
GREY   = "#9FADB8"   # neutral context (non-highlighted)
GRID   = "#E9ECEF"   # gridlines
CAT    = [BLUE, TEAL, AMBER, CYAN, SLATE]           # fixed categorical order
DIAG   = {"M": BLUE, "B": TEAL}                      # semantic: Malignant=Blue, Benign=Teal

def apply_style():
    mpl.rcParams.update({
        "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica Neue", "DejaVu Sans"],
        "font.size": 10.5, "axes.titlesize": 12.5, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.titlepad": 10,
        "axes.edgecolor": SLATE, "axes.linewidth": 0.8, "axes.labelcolor": NAVY,
        "axes.spines.top": False, "axes.spines.right": False,
        "text.color": NAVY, "xtick.color": NAVY, "ytick.color": NAVY,
        "grid.color": GRID, "grid.linewidth": 0.8, "axes.grid": True, "axes.grid.axis": "y",
        "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    })

apply_style()

import hashlib, json, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
SRC = ROOT / "data.csv"
print("project root:", ROOT.name)
print("source exists:", SRC.exists())

project root: Breast Cancer Wisconsin
source exists: True


## 1.1 Load the raw file (comma-delimited, has a header)

The delivered `data.csv` has a header row and a **trailing comma**, which makes pandas materialise a
phantom 33rd column. We load *as delivered* first, inspect, then handle it — nothing is silently coerced.

In [2]:
raw = pd.read_csv(SRC)
print("raw shape:", raw.shape)
print("columns:", list(raw.columns))
raw.head(3)

raw shape: (569, 33)
columns: ['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32']


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN


## 1.2 Drop the phantom all-null column (trailing-comma artifact)

In [3]:
null_frac = raw.isna().mean()
phantom = null_frac[null_frac == 1.0].index.tolist()
print("fully-null columns:", phantom)
assert all(raw[c].isna().all() for c in phantom), "a 'phantom' column is not 100% null — investigate"

df = raw.drop(columns=phantom)
print(f"dropped {len(phantom)} phantom column(s) -> usable shape: {df.shape}")
assert df.shape[1] == 32, f"expected 32 usable columns (id + diagnosis + 30 features), got {df.shape[1]}"

fully-null columns: ['Unnamed: 32']
dropped 1 phantom column(s) -> usable shape: (569, 32)


## 1.3 Lineage log — immutable raw copy + hash

Raw data is never edited after ingestion. We save an untouched copy to `data/raw/` and record source,
SHA-256, pull timestamp, and shapes to `_lineage.csv`.

In [4]:
sha256 = hashlib.sha256(SRC.read_bytes()).hexdigest()
raw.to_csv(RAW_DIR / "data.csv", index=False)  # immutable copy, as delivered

lineage = pd.DataFrame([{
    "source_file": SRC.name,
    "sha256": sha256,
    "pull_timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "raw_rows": raw.shape[0], "raw_cols": raw.shape[1],
    "usable_rows": df.shape[0], "usable_cols": df.shape[1],
    "phantom_cols_dropped": len(phantom),
}])
lineage.to_csv(RAW_DIR / "_lineage.csv", index=False)
print("SHA-256:", sha256)
lineage.T

SHA-256: 1425d9affa78ba8e53afc81d0ef8a19069ee10c4b21fe89b3cf514071b12ee33


,0
source_file,data.csv
sha256,1425d9affa78ba8e53afc81d0ef8a19069ee10c4b21fe8...
pull_timestamp_utc,2026-07-20T15:43:55+00:00
raw_rows,569
raw_cols,33
usable_rows,569
usable_cols,32
phantom_cols_dropped,1


## 1.4 Schema validation against the data dictionary

Confirm dtypes, expected ranges, and — critically — the **published WDBC class split (357 B / 212 M)**.
If the split differs, this file is not the canonical dataset and everything downstream is suspect.

In [5]:
BASE = ["radius","texture","perimeter","area","smoothness","compactness",
        "concavity","concave points","symmetry","fractal_dimension"]
FEATURES = [f"{b}_{s}" for s in ["mean","se","worst"] for b in BASE]

checks = []
def chk(name, ok, detail=""):
    checks.append({"rule": name, "pass": bool(ok), "detail": str(detail)})

chk("id present & integer", "id" in df.columns)
chk("diagnosis in {M,B}", set(df["diagnosis"].unique()) <= {"M","B"}, sorted(df["diagnosis"].unique()))
chk("30 feature columns present", all(f in df.columns for f in FEATURES),
    f"{sum(f in df.columns for f in FEATURES)}/30 found")
chk("all features numeric", all(pd.api.types.is_numeric_dtype(df[f]) for f in FEATURES if f in df.columns))
chk("all features non-negative", bool((df[FEATURES] >= 0).all().all()))
_zero_cols = (df[FEATURES] == 0).sum(); _zero_cols = _zero_cols[_zero_cols > 0]
chk("zeros only in concavity/concave-points (zero-inflation, expected)",
    set(_zero_cols.index) <= {f"{b}_{s}" for b in ["concavity","concave points"] for s in ["mean","se","worst"]},
    f"{len(_zero_cols)} cols, {int(_zero_cols.max()) if len(_zero_cols) else 0} rows each")
chk("area_mean plausible (<=3000)", df["area_mean"].max() <= 3000, f"max={df['area_mean'].max()}")
chk("smoothness_mean in [0.03,0.20]", df["smoothness_mean"].between(0.03,0.20).all(),
    f"[{df['smoothness_mean'].min():.4f}, {df['smoothness_mean'].max():.4f}]")

vc = df["diagnosis"].value_counts()
chk("class split == 357 B / 212 M", vc.get("B")==357 and vc.get("M")==212, dict(vc))
chk("no duplicate id", df["id"].duplicated().sum()==0, f"{df['id'].duplicated().sum()} dupes")

schema = pd.DataFrame(checks)
schema.to_csv(RAW_DIR / "_schema_validation.csv", index=False)
print("all schema rules pass:", bool(schema["pass"].all()))
schema

all schema rules pass: True


,rule,pass,detail
0,id present & integer,True,
1,"diagnosis in {M,B}",True,"['B', 'M']"
2,30 feature columns present,True,30/30 found
3,all features numeric,True,
4,all features non-negative,True,
5,zeros only in concavity/concave-points (zero-i...,True,"6 cols, 13 rows each"
6,area_mean plausible (<=3000),True,max=2501.0
7,"smoothness_mean in [0.03,0.20]",True,"[0.0526, 0.1634]"
8,class split == 357 B / 212 M,True,"{'B': np.int64(357), 'M': np.int64(212)}"
9,no duplicate id,True,0 dupes


## 1.5 Governance classification

De-identified research data: `id` is an opaque key with **no linkage, no demographics, no PII**. This is
recorded here and carried into the Cross-Cutting section (notebook 04) — where the *absence* of any
demographic attribute becomes the reason a fairness subgroup audit is **impossible** and the deployment
claim must be capped accordingly.

In [6]:
gov = pd.DataFrame([
    {"field":"id","classification":"opaque key","pii":"no","action":"drop in cleaning (row-key only)"},
    {"field":"diagnosis","classification":"clinical label (target)","pii":"no","action":"encode M=1/B=0"},
    {"field":"30 morphology features","classification":"derived image measurements","pii":"no","action":"retain"},
    {"field":"demographics (age/race/site)","classification":"ABSENT","pii":"n/a",
     "action":"none available -> fairness subgroup audit impossible (documented in 04)"},
])
gov.to_csv(RAW_DIR / "_governance.csv", index=False)
gov

,field,classification,pii,action
0,id,opaque key,no,drop in cleaning (row-key only)
1,diagnosis,clinical label (target),no,encode M=1/B=0
2,30 morphology features,derived image measurements,no,retain
3,demographics (age/race/site),ABSENT,n/a,none available -> fairness subgroup audit impo...


## Stage 1 — Gate Checklist

- [x] **Raw data saved to `data/raw/`** — untouched copy + SHA-256 recorded
- [x] **Source, timestamp, and row/column counts logged** — `_lineage.csv` (569×33 raw → 569×32 usable)
- [x] **Schema matches expectations** — dtypes, positive-value & range checks, and the **357 B / 212 M** class split all pass (`_schema_validation.csv`)
- [x] **PII / governance review completed** — de-identified; no demographics present (the fairness constraint is logged for notebook 04)

**→ Proceed to `02_cleaning.ipynb`.**